# 0412 · Modelos base, versión 2: referencias y lineal / logística

> **Versión 2 (05-11-2026, evaluación del director).** Lee `V4/MODELADO/` (0411 v2), comprueba la huella de la base (A6) y agrega referencias:
> - **C3 (regresión):** mediana de horas por zona×franja del entrenamiento y regresión de cuantil 0,5 (`QuantileRegressor`). Así un ensamble con pérdida L1 se compara con referencias de mediana, y uno con pérdida L2 con las de media.
> - **A5 (clasificación):** tasa histórica de incumplimiento de la zona×franja (sin futuro) y la regla operativa «menor margen primero».

| Modelo | Clasificación (`clf`) | Regresión (`reg`) |
| --- | --- | --- |
| `Referencia` | Prevalencia del entrenamiento, igual para todas | Media histórica de horas de su zona×franja (sin futuro) |
| `Referencia_tasa_historica` | Tasa histórica de incumplimiento de su zona×franja (sin futuro) | |
| `Referencia_menor_margen` | Menos margen hasta el vencimiento, más riesgo (solo ordena) | |
| `Referencia_mediana` | | Mediana de horas de su zona×franja en el entrenamiento |
| `Lineal` | Regresión logística | Regresión lineal (media) |
| `Lineal_mediana` | | Regresión de cuantil 0,5 (mediana) |

`Referencia_menor_margen` es un puntaje para ordenar guías, no una probabilidad: sirve para AP, AUC y alertas con capacidad fija, pero su Brier y su pérdida logarítmica no se interpretan.

**No depende de los otros cuadernos de modelos.** Solo necesita 0411 v2. No hay prueba final aquí.

In [ ]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [ ]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

## 0. Configuración

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import QuantileRegressor
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas del espacio de trabajo

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, verificar_huella, pliegues_origen_movil, datos_del_pliegue,
                                       marco_predicciones, construir_modelo, predecir, tabla_metricas,
                                       mediana_por_grupo, puntaje_menor_margen)

In [ ]:
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
MAX_FILAS_CUANTIL = 30_000    # submuestra para la regresión de cuantil: el solver crece casi con n² (200.000 filas tardaban horas)

## 1. Cargar la preparación de 0411

In [ ]:
print("Huella de la base comprobada:", verificar_huella(CARPETA_MODELADO)[:16], "...")   # A6
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · bloques: {CFG['INICIOS_VALIDACION']} · preparado el {CFG['fecha_preparacion']}")

## 2. Validación de origen móvil

En cada versión de variables, tarea y bloque: se arma el entrenamiento (antes del bloque) y la validación (el bloque), se calculan las referencias y se entrenan los modelos lineales. Las referencias solo usan el entrenamiento o la historia sin futuro de 0411.

In [ ]:
pred_ref, pred_lin = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    cols = CATEGORICAS + numericas
    for tarea, y_col, _ in TAREAS:
        for ini, tr, va in PLIEGUES[tarea]:
            d, tr, va = datos_del_pliegue(df, HIST[ini], y_col, tr, va)
            Dtr, Dva = d.iloc[tr], d.iloc[va]
            Xtr, ytr, Xva, yva = Dtr[cols], Dtr[y_col], Dva[cols], Dva[y_col]
            base = marco_predicciones(version, tarea, ini, Dva, yva)
            if tarea == "clf":
                pred_ref.append(base.assign(modelo="Referencia", pred=np.full(len(Dva), ytr.mean())))
                pred_ref.append(base.assign(modelo="Referencia_tasa_historica",
                                            pred=Dva["hist_inc_zona_franja"].fillna(ytr.mean()).values))
                pred_ref.append(base.assign(modelo="Referencia_menor_margen",
                                            pred=puntaje_menor_margen(Dva["margen_sla_en_t0_h"].values)))
            else:
                pred_ref.append(base.assign(modelo="Referencia", pred=Dva["hist_h_zona_franja"].fillna(ytr.mean()).values))
                pred_ref.append(base.assign(modelo="Referencia_mediana",
                                            pred=mediana_por_grupo(ytr.values, Dtr["zona_franja"].values, Dva["zona_franja"].values)))
            # Lineal (regresión lineal o logística según la tarea)
            m = construir_modelo(tarea, "Lineal", numericas, CATEGORICAS, {}, SEMILLA).fit(Xtr, ytr)
            pred_lin.append(base.assign(modelo="Lineal", pred=predecir(m, Xva, tarea)))
            if tarea == "reg" and version == "internas":   # C3: lineal de mediana (cuantil 0,5); solo internas, que es donde se usa
                sub = np.arange(len(Xtr))
                if len(sub) > MAX_FILAS_CUANTIL:
                    sub = np.sort(np.random.default_rng(SEMILLA).choice(sub, MAX_FILAS_CUANTIL, replace=False))
                mq = construir_modelo("reg", "Lineal", numericas, CATEGORICAS, {}, SEMILLA)
                mq.set_params(modelo=QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs"))
                mq.fit(Xtr.iloc[sub], ytr.iloc[sub])
                pred_lin.append(base.assign(modelo="Lineal_mediana", pred=predecir(mq, Xva, "reg")))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred_ref, pred_lin = pd.concat(pred_ref, ignore_index=True), pd.concat(pred_lin, ignore_index=True)

## 3. Vista rápida

Solo para revisar los modelos base. La comparación con los ensambles se hace en 0416.

In [ ]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo"]).round(4))

In [ ]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo", "bloque"]).round(4))

## 4. Guardar para 0416 v2

In [ ]:
pred_ref.to_parquet(f"{CARPETA_MODELADO}pred_val_Referencia.parquet", index=False)
pred_lin.to_parquet(f"{CARPETA_MODELADO}pred_val_Lineal.parquet", index=False)
print("Guardado en", CARPETA_MODELADO)